# Test Classification Generator

Generate dialect predictions for the 579 test audio files using the trained TamilWav2VecLearnedAttentive model.

**Output Format:** `<test_file_id> <dialect_code>` (space-separated, one per line)

In [1]:
import os
import torch
import librosa
import numpy as np
from tqdm.notebook import tqdm
from transformers import Wav2Vec2FeatureExtractor
from model import TamilWav2VecLearnedAttentive

In [2]:
# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Paths
CHECKPOINT_PATH = "train_data/checkpoints_1/best_model.pth"
TEST_DIR = "../../Final Dataset/Test"
OUTPUT_PATH = "../../Final Results/TEAM_NAME_Classification_Run1.txt"

# Model config
MODEL_NAME = "Harveenchadha/vakyansh-wav2vec2-tamil-tam-250"
SAMPLING_RATE = 16000

# Dialect mapping to submission codes
# 0: Central_Dialect (Kongu) -> Central
# 1: Northern_Dialect (Chennai) -> Northern
# 2: Southern_Dialect (Srilankan) -> Southern
# 3: Western_Dialect (Thanjavur) -> Western
DIALECT_MAP = {
    0: "Central",
    1: "Northern",
    2: "Southern",
    3: "Western"
}

Using device: cuda


In [3]:
# Load model
print("Loading model...")
model = TamilWav2VecLearnedAttentive(
    model_name=MODEL_NAME,
    num_classes=4,
    num_unfrozen_layers=4
)

# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()
print("Model loaded successfully!")

# Load feature extractor
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
print("Feature extractor loaded!")

Loading model...
Model loaded successfully!
Feature extractor loaded!


In [4]:
# Get all test files
test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.wav')])
print(f"Found {len(test_files)} test files")

Found 579 test files


In [5]:
# Process each file one by one
results = []

with torch.no_grad():
    for filename in tqdm(test_files, desc="Processing test files"):
        audio_path = os.path.join(TEST_DIR, filename)
        audio_id = os.path.splitext(filename)[0]  # e.g., test_0001
        
        try:
            # Load audio
            waveform, sr = librosa.load(audio_path, sr=SAMPLING_RATE, mono=True)
            
            # Extract features
            inputs = feature_extractor(
                waveform,
                sampling_rate=SAMPLING_RATE,
                return_tensors='pt',
                padding=True,
                return_attention_mask=True,
                do_normalize=False
            )
            
            input_values = inputs['input_values'].to(DEVICE)
            attention_mask = inputs['attention_mask'].to(DEVICE)
            
            # Get predictions
            logits = model(input_values, attention_mask)
            predicted_class = logits.argmax(dim=-1).item()
            dialect_code = DIALECT_MAP[predicted_class]
            
            results.append(f"{audio_id} {dialect_code}")
            
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            # Default to Central on error (shouldn't happen)
            results.append(f"{audio_id} Central")

print(f"\nProcessed {len(results)} files")

Processing test files:   0%|          | 0/579 [00:00<?, ?it/s]


Processed 579 files


In [6]:
# Save results in submission format
with open(OUTPUT_PATH, 'w') as f:
    f.write('\n'.join(results))

print(f"Results saved to: {OUTPUT_PATH}")

# Show distribution
from collections import Counter
dialect_counts = Counter([r.split()[1] for r in results])
print("\nDialect Distribution:")
for dialect, count in sorted(dialect_counts.items()):
    print(f"  {dialect}: {count}")

Results saved to: ../../Final Results/TEAM_NAME_Classification_Run1.txt

Dialect Distribution:
  Central: 57
  Northern: 240
  Southern: 257
  Western: 25


In [7]:
# Preview first 10 results
print("Preview (first 10 lines):")
for line in results[:10]:
    print(line)

Preview (first 10 lines):
test_0001 Northern
test_0002 Southern
test_0003 Southern
test_0004 Northern
test_0005 Southern
test_0006 Southern
test_0007 Southern
test_0008 Southern
test_0009 Central
test_0010 Northern
